# Secretory Lineage scRNA-seq BBKNN Analysis with Marker Identification

**Purpose**: BBKNN-based reclustering and comprehensive marker gene analysis

**Key Features**:
1. Small batch filtering (following QUICK_REFERENCE_MEMORY best practices)
2. BBKNN clustering with multiple resolutions
3. Marker gene identification per cluster and CellTypist annotation
4. Comprehensive visualization

**Input**: Secretory_Lineage_filtered.h5ad

**Author**: r2end
**Date**: 2025-01-16
**Version**: v1.0



In [ ]:
# ===== Configuration =====
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from scipy import sparse
from typing import Optional, Dict, List, Tuple
import logging

# Check bbknn availability
try:
    import bbknn
    print(f"✓ bbknn version: {bbknn.__version__ if hasattr(bbknn, '__version__') else 'unknown'}")
except ImportError:
    raise ImportError("bbknn not installed. Install: pip install bbknn")

# Set plotting parameters
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.set_figure_params(scanpy=True, dpi=100, dpi_save=300, 
                      vector_friendly=True, fontsize=12)
sc.settings.n_jobs = 48  # Utilize multi-core

# File paths
INPUT_FILE = '/home/h2048/data/py/0114/cnmf_v1.3_fixed_optimized_k/Secretory_Lineage_filtered.h5ad'
OUTPUT_DIR = '/home/h2048/data/py/0114/cnmf_v1.3_fixed_optimized_k/bbknn_analysis/'

# Create output directory
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Random seed for reproducibility
np.random.seed(42)

# BBKNN parameters
BBKNN_CONFIG = {
    'batch_key': None,  # Will auto-detect
    'n_pcs': 50,
    'neighbors_within_batch': 7,
    'metric': 'euclidean',
    'trim': None,
    'min_cells_per_batch': 10,  # Filter small batches
    'min_batches': 2,
    'leiden_resolutions': [0.5, 1.0, 1.5],
    'default_resolution': 1.5,
    'umap_min_dist': 0.3,
    'umap_spread': 1.0,
}

# Marker analysis parameters
MARKER_CONFIG = {
    'min_pct': 0.25,
    'logfc_threshold': 0.25,
    'top_n': 20,
    'method': 'wilcoxon',  # Fast for exploratory
    'use_raw': True,
}

print(f"\n{'='*80}")
print("SECRETORY LINEAGE BBKNN ANALYSIS")
print(f"{'='*80}\n")



## 1. Data Loading and Structure Inspection



In [ ]:
print(f"{'='*80}")
print("STEP 1: DATA LOADING")
print(f"{'='*80}\n")

# Load data
adata = sc.read_h5ad(INPUT_FILE)

print(f"✓ Data loaded successfully")
print(f"  Shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

# Memory usage
if sparse.issparse(adata.X):
    mem_gb = adata.X.data.nbytes / 1e9
    print(f"  Memory: {mem_gb:.2f} GB (sparse)")
else:
    mem_gb = adata.X.nbytes / 1e9
    print(f"  Memory: {mem_gb:.2f} GB (dense)")

# Check data structure
print(f"\nData structure:")
print(f"  .X type: {type(adata.X).__name__}")
print(f"  .layers: {list(adata.layers.keys())}")
print(f"  .obsm: {list(adata.obsm.keys())}")
print(f"  .raw: {'Available' if adata.raw else 'Not available'}")

# Check required layer
if 'counts' not in adata.layers and 'log1p' not in adata.layers:
    raise ValueError("Missing required layer: 'counts' or 'log1p'")

print(f"\n✓ Data structure verified")



In [ ]:
print(f"\n{'='*80}")
print("STEP 4: BATCH KEY DETECTION")
print(f"{'='*80}\n")

# Auto-detect batch key
batch_key_candidates = ['dataset', 'batch', 'Sample', 'sample', 'sample_id', 'orig.ident']
batch_key = None

for candidate in batch_key_candidates:
    if candidate in adata.obs.columns:
        n_batches = adata.obs[candidate].nunique()
        if n_batches >= BBKNN_CONFIG['min_batches']:
            batch_key = candidate
            print(f"✓ Detected batch key: '{batch_key}' ({n_batches} batches)")
            break

if batch_key is None:
    raise ValueError(f"No valid batch key found. Tried: {batch_key_candidates}")

BBKNN_CONFIG['batch_key'] = batch_key

# Batch composition analysis
batch_counts = adata.obs[batch_key].value_counts().sort_values(ascending=False)
print(f"\nBatch composition:")
print(f"  Total batches: {len(batch_counts)}")
print(f"  Cells per batch (top 10):")
for batch, count in batch_counts.head(10).items():
    print(f"    {batch}: {count:,} cells")
if len(batch_counts) > 10:
    print(f"    ... and {len(batch_counts) - 10} more batches")



In [ ]:
print(f"\n{'='*80}")
print("STEP 5: SMALL BATCH FILTERING")
print(f"{'='*80}\n")

# Identify small batches
small_batches = batch_counts[batch_counts < BBKNN_CONFIG['min_cells_per_batch']]

if len(small_batches) > 0:
    print(f"⚠️  Found {len(small_batches)} small batches (< {BBKNN_CONFIG['min_cells_per_batch']} cells):")
    for batch, count in small_batches.items():
        print(f"    {batch}: {count} cells")
    
    # Filter
    valid_batches = batch_counts[batch_counts >= BBKNN_CONFIG['min_cells_per_batch']].index
    n_valid = len(valid_batches)
    
    if n_valid < BBKNN_CONFIG['min_batches']:
        raise ValueError(f"Too few valid batches after filtering ({n_valid} < {BBKNN_CONFIG['min_batches']})")
    
    print(f"\n  Filtering to {n_valid} valid batches...")
    adata = adata[adata.obs[batch_key].isin(valid_batches)].copy()
    print(f"  Retained: {adata.n_obs:,} cells")
    
    # Update batch counts
    batch_counts = adata.obs[batch_key].value_counts()
else:
    print(f"✓ All batches have sufficient cells (≥{BBKNN_CONFIG['min_cells_per_batch']})")

# Auto-adjust neighbors_within_batch
min_batch_size = batch_counts.min()
neighbors_within = BBKNN_CONFIG['neighbors_within_batch']

if neighbors_within >= min_batch_size:
    original_neighbors = neighbors_within
    neighbors_within = max(1, min_batch_size - 1)
    print(f"\n⚠️  Auto-adjusting neighbors_within_batch:")
    print(f"    {original_neighbors} → {neighbors_within} (min batch size: {min_batch_size})")
    BBKNN_CONFIG['neighbors_within_batch'] = neighbors_within



In [ ]:
# ===== GENE FILTERING: Remove Ribosomal, Mitochondrial, and Unmapped ENSG Genes =====
print(f"\n{'='*80}")
print("GENE FILTERING: Ribosomal, Mitochondrial, and ENSG Genes")
print(f"{'='*80}\n")

# Record initial state
n_genes_before = adata.n_vars
n_cells_before = adata.n_obs

print(f"Initial dataset:")
print(f"  Cells: {n_cells_before:,}")
print(f"  Genes: {n_genes_before:,}")

# Get gene names
gene_names = adata.var_names.astype(str)

# ===== 1. Identify genes to remove =====
print(f"\n{'='*80}")
print(f"Identifying genes to filter...")
print(f"{'='*80}\n")

# Mitochondrial genes (MT-)
mt_genes = gene_names.str.startswith('MT-') | gene_names.str.startswith('mt-')
n_mt = mt_genes.sum()
print(f"1. Mitochondrial genes (MT-*):")
print(f"   Found: {n_mt:,} genes")
if n_mt > 0 and n_mt <= 20:
    mt_examples = gene_names[mt_genes].tolist()[:10]
    print(f"   Examples: {', '.join(mt_examples)}")

# Ribosomal genes (RPL*, RPS*)
ribo_genes = (
    gene_names.str.startswith('RPL') | 
    gene_names.str.startswith('RPS') |
    gene_names.str.startswith('rpl') |
    gene_names.str.startswith('rps')
)
n_ribo = ribo_genes.sum()
print(f"\n2. Ribosomal genes (RPL*, RPS*):")
print(f"   Found: {n_ribo:,} genes")
if n_ribo > 0 and n_ribo <= 20:
    ribo_examples = gene_names[ribo_genes].tolist()[:10]
    print(f"   Examples: {', '.join(ribo_examples)}")

# ENSG genes (unmapped ENSEMBL IDs)
ensg_genes = gene_names.str.startswith('ENSG')
n_ensg = ensg_genes.sum()
print(f"\n3. Unmapped ENSEMBL IDs (ENSG*):")
print(f"   Found: {n_ensg:,} genes")
if n_ensg > 0 and n_ensg <= 20:
    ensg_examples = gene_names[ensg_genes].tolist()[:10]
    print(f"   Examples: {', '.join(ensg_examples)}")

# Combined filter
genes_to_remove = mt_genes | ribo_genes | ensg_genes
n_to_remove = genes_to_remove.sum()
n_to_keep = n_genes_before - n_to_remove

print(f"\n{'='*80}")
print(f"Filtering Summary:")
print(f"{'='*80}")
print(f"  Total genes to remove: {n_to_remove:,} ({n_to_remove/n_genes_before*100:.1f}%)")
print(f"  - Mitochondrial: {n_mt:,}")
print(f"  - Ribosomal: {n_ribo:,}")
print(f"  - ENSG unmapped: {n_ensg:,}")
print(f"  Genes to keep: {n_to_keep:,} ({n_to_keep/n_genes_before*100:.1f}%)")

# Safety check
if n_to_keep < 5000:
    print(f"\n⚠️  WARNING: Only {n_to_keep:,} genes will remain after filtering!")
    print(f"   This might be too aggressive. Consider reviewing the filter criteria.")
    
    response = input(f"\n   Proceed with filtering? (yes/no): ").strip().lower()
    if response not in ['yes', 'y']:
        print(f"   Filtering cancelled by user.")
        # Skip filtering but continue
        genes_to_remove = np.zeros(n_genes_before, dtype=bool)
        n_to_remove = 0
        n_to_keep = n_genes_before

# ===== 2. Save gene lists before filtering =====
print(f"\n{'='*80}")
print(f"Saving filtered gene lists...")
print(f"{'='*80}\n")

# Create output directory if needed
filter_output_dir = OUTPUT_DIR + 'gene_filtering/'
os.makedirs(filter_output_dir, exist_ok=True)

# Save removed genes
removed_genes_df = pd.DataFrame({
    'gene_name': gene_names[genes_to_remove],
    'category': ['Mitochondrial' if mt else 'Ribosomal' if ribo else 'ENSG_unmapped'
                 for mt, ribo in zip(mt_genes[genes_to_remove], 
                                    ribo_genes[genes_to_remove])]
})
removed_file = filter_output_dir + 'removed_genes.csv'
removed_genes_df.to_csv(removed_file, index=False)
print(f"✓ Removed genes saved: {removed_file}")

# Save kept genes
kept_genes_df = pd.DataFrame({
    'gene_name': gene_names[~genes_to_remove]
})
kept_file = filter_output_dir + 'kept_genes.csv'
kept_genes_df.to_csv(kept_file, index=False)
print(f"✓ Kept genes saved: {kept_file}")

# ===== 3. Apply filter =====
if n_to_remove > 0:
    print(f"\n{'='*80}")
    print(f"Applying gene filter...")
    print(f"{'='*80}\n")
    
    # Keep genes
    adata = adata[:, ~genes_to_remove].copy()
    
    print(f"✓ Filtering completed")
    print(f"\n  Dataset after filtering:")
    print(f"    Cells: {adata.n_obs:,} (unchanged)")
    print(f"    Genes: {adata.n_vars:,} (removed {n_to_remove:,})")
    
    # Verify
    assert adata.n_vars == n_to_keep, "Gene count mismatch after filtering!"
    
    # Check remaining genes
    remaining_genes = adata.var_names.astype(str)
    still_has_mt = remaining_genes.str.startswith('MT-').sum()
    still_has_ribo = (remaining_genes.str.startswith('RPL') | 
                      remaining_genes.str.startswith('RPS')).sum()
    still_has_ensg = remaining_genes.str.startswith('ENSG').sum()
    
    if still_has_mt > 0 or still_has_ribo > 0 or still_has_ensg > 0:
        print(f"\n  ⚠️  Warning: Some filtered genes still present:")
        print(f"     MT-: {still_has_mt}, Ribosomal: {still_has_ribo}, ENSG: {still_has_ensg}")
    else:
        print(f"\n  ✓ Verification: All target genes removed successfully")
    
    # Memory cleanup
    gc.collect()
    
else:
    print(f"\n✓ No genes to filter (skipping)")

# ===== 4. Gene statistics after filtering =====
print(f"\n{'='*80}")
print(f"Gene Statistics After Filtering:")
print(f"{'='*80}\n")

# Gene name patterns
remaining_genes = adata.var_names.astype(str)

# Count different gene types
gene_types = {
    'Standard genes': ~(remaining_genes.str.contains('ENSG|MT-|RPL|RPS|^RP[0-9]', case=False)),
    'Pseudogenes (*P)': remaining_genes.str.contains('P[0-9]+$', regex=True),
    'lncRNA (LINC*, etc)': remaining_genes.str.startswith('LINC'),
    'Other non-coding': remaining_genes.str.contains('^[A-Z]{2}[0-9]{6}', regex=True),
}

print(f"Remaining gene categories:")
for cat, mask in gene_types.items():
    n = mask.sum()
    pct = n / adata.n_vars * 100
    print(f"  {cat}: {n:,} ({pct:.1f}%)")

# Check for common markers (sanity check)
print(f"\n{'='*80}")
print(f"Sanity Check: Common Marker Genes")
print(f"{'='*80}\n")

common_markers = [
    'EPCAM', 'KRT5', 'TP63',  # Epithelial
    'SCGB1A1', 'MUC5AC', 'MUC5B',  # Secretory
    'FOXJ1', 'CD3D', 'CD3E',  # Ciliated, T cells
    'PTPRC', 'CD79A', 'MS4A1',  # Immune
    'COL1A1', 'DCN', 'VWF'  # Stromal, Endothelial
]

found_markers = [m for m in common_markers if m in adata.var_names]
missing_markers = [m for m in common_markers if m not in adata.var_names]

print(f"Common markers found: {len(found_markers)}/{len(common_markers)}")
if found_markers:
    print(f"  Present: {', '.join(found_markers)}")
if missing_markers:
    print(f"  Missing: {', '.join(missing_markers)}")
    print(f"  (This is OK if they weren't in the original data)")

# ===== 5. Summary =====
print(f"\n{'='*80}")
print(f"GENE FILTERING COMPLETE")
print(f"{'='*80}\n")

filtering_summary = {
    'initial_cells': n_cells_before,
    'initial_genes': n_genes_before,
    'final_cells': adata.n_obs,
    'final_genes': adata.n_vars,
    'removed_mt': n_mt,
    'removed_ribo': n_ribo,
    'removed_ensg': n_ensg,
    'total_removed': n_to_remove,
    'removal_rate': f"{n_to_remove/n_genes_before*100:.1f}%"
}

for key, value in filtering_summary.items():
    print(f"  {key}: {value}")

# Save summary
summary_file = filter_output_dir + 'filtering_summary.json'
import json
with open(summary_file, 'w') as f:
    json.dump(filtering_summary, f, indent=2)
print(f"\n✓ Summary saved: {summary_file}")

# Update uns metadata
adata.uns['gene_filtering'] = filtering_summary

print(f"\n{'='*80}")
print(f"✓ Ready for normalization and downstream analysis")
print(f"{'='*80}")

## 2. Normalization Check and Correction



In [ ]:
print(f"\n{'='*80}")
print("STEP 2: NORMALIZATION CHECK")
print(f"{'='*80}\n")

# Check if data is normalized
def check_normalization(adata, layer_key=None):
    """
    Check if data appears to be log-normalized.
    Returns: (is_normalized, max_value)
    """
    if layer_key:
        X = adata.layers[layer_key]
    else:
        X = adata.X
    
    if sparse.issparse(X):
        max_val = X.data.max()
    else:
        max_val = X.max()
    
    # Log-normalized data typically has max value < 15
    is_normalized = max_val < 15
    return is_normalized, max_val

# Check current state
is_norm, max_val = check_normalization(adata)

print(f"Current .X state:")
print(f"  Max value: {max_val:.2f}")
print(f"  Appears log-normalized: {is_norm}")

# If not normalized, check for count layer and normalize
if not is_norm:
    print(f"\n⚠️  .X does not appear log-normalized (max={max_val:.1f})")
    
    if 'counts' in adata.layers:
        print(f"  Found 'counts' layer, will normalize...")
        
        # Backup current X if needed
        if 'X_raw' not in adata.layers:
            adata.layers['X_raw'] = adata.X.copy()
        
        # Normalize from counts
        adata.X = adata.layers['counts'].copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        
        # Save to log1p layer
        adata.layers['log1p'] = adata.X.copy()
        
        print(f"✓ Normalized: target_sum=1e4, log1p applied")
        
        # Verify
        is_norm, max_val = check_normalization(adata)
        print(f"  New max value: {max_val:.2f}")
        
    elif 'log1p' in adata.layers:
        print(f"  Found 'log1p' layer, using it for .X...")
        adata.X = adata.layers['log1p'].copy()
        
        is_norm, max_val = check_normalization(adata)
        print(f"✓ Loaded log1p layer")
        print(f"  Max value: {max_val:.2f}")
    else:
        raise ValueError("Cannot find 'counts' or 'log1p' layer for normalization")
else:
    print(f"✓ Data is already log-normalized")



In [ ]:
# ===== VERIFICATION: Confirm Normalized Data Before HVG =====
print(f"\n{'='*80}")
print("VERIFICATION: Data Normalization Check (Before HVG)")
print(f"{'='*80}\n")

# Check adata.X normalization state
print(f"Checking adata.X:")
if sparse.issparse(adata.X):
    x_max = adata.X.data.max()
    x_min = adata.X.data.min()
    x_mean = adata.X.data.mean()
else:
    x_max = adata.X.max()
    x_min = adata.X.min()
    x_mean = adata.X.mean()

print(f"  Min value: {x_min:.3f}")
print(f"  Max value: {x_max:.3f}")
print(f"  Mean value: {x_mean:.3f}")

# Assessment
if x_max < 15:
    print(f"  ✓ PASS: adata.X appears log-normalized (max < 15)")
    x_status = "log-normalized"
elif x_max < 50:
    print(f"  ⚠️  WARNING: adata.X borderline (15 < max < 50)")
    x_status = "borderline"
else:
    print(f"  ❌ FAIL: adata.X appears to be raw counts (max > 50)")
    x_status = "counts"

# Check layers
print(f"\nAvailable layers: {list(adata.layers.keys())}")

if 'counts' in adata.layers:
    if sparse.issparse(adata.layers['counts']):
        counts_max = adata.layers['counts'].data.max()
    else:
        counts_max = adata.layers['counts'].max()
    print(f"  layers['counts'] max value: {counts_max:.1f}")
    
    if counts_max > 50:
        print(f"    ✓ Appears to be raw counts (good for HVG)")
        counts_status = "raw_counts"
    else:
        print(f"    ⚠️  WARNING: Might not be raw counts!")
        counts_status = "normalized"
else:
    print(f"  ⚠️  WARNING: No 'counts' layer found!")
    counts_status = "missing"

if 'log1p' in adata.layers:
    if sparse.issparse(adata.layers['log1p']):
        log1p_max = adata.layers['log1p'].data.max()
    else:
        log1p_max = adata.layers['log1p'].max()
    print(f"  layers['log1p'] max value: {log1p_max:.1f}")

# Sample a few genes to show distribution
print(f"\n{'='*80}")
print(f"Sample Gene Expression Check:")
print(f"{'='*80}")

sample_genes = adata.var_names[:5].tolist()  # First 5 genes
print(f"\nChecking first 5 genes: {', '.join(sample_genes)}")
print(f"\n{'Gene':<15} {'X_max':>10} {'X_mean':>10} {'Assessment':>15}")
print(f"{'-'*52}")

all_x_ok = True
for gene in sample_genes:
    idx = adata.var_names.get_loc(gene)
    if sparse.issparse(adata.X):
        expr = adata.X[:, idx].toarray().flatten()
    else:
        expr = adata.X[:, idx].flatten()
    
    gene_max = expr.max()
    gene_mean = expr.mean()
    
    if gene_max < 15:
        status = "✓ OK"
    elif gene_max < 50:
        status = "⚠️  Borderline"
        all_x_ok = False
    else:
        status = "❌ Too high"
        all_x_ok = False
    
    print(f"{gene:<15} {gene_max:>10.2f} {gene_mean:>10.3f} {status:>15}")

# Final recommendation
print(f"\n{'='*80}")
print(f"FINAL ASSESSMENT:")
print(f"{'='*80}")

if x_status == "log-normalized" and counts_status == "raw_counts":
    print(f"✓ OPTIMAL: adata.X is log-normalized, layers['counts'] is raw")
    print(f"  → HVG will use layers['counts'] (correct)")
    print(f"  → Visualization will use adata.X (correct)")
    final_status = "PASS"
elif x_status == "log-normalized" and counts_status == "missing":
    print(f"⚠️  SUBOPTIMAL: adata.X is log-normalized but no counts layer")
    print(f"  → HVG will use adata.X (may work but not ideal)")
    print(f"  → Consider if original counts are available")
    final_status = "WARNING"
elif x_status == "counts":
    print(f"❌ PROBLEM: adata.X appears to be raw counts!")
    print(f"  → This will cause issues in visualization later")
    print(f"  → Should run normalization step")
    final_status = "FAIL"
else:
    print(f"⚠️  UNCLEAR: Data structure is ambiguous")
    print(f"  → Proceed with caution and check results carefully")
    final_status = "WARNING"

print(f"\nStatus: {final_status}")

if final_status == "FAIL":
    print(f"\n❌ STOP: Fix normalization before proceeding!")
elif final_status == "WARNING":
    print(f"\n⚠️  CAUTION: Results may need verification")
else:
    print(f"\n✓ OK to proceed with HVG calculation")

## 3. Highly Variable Genes with Forced Markers



In [ ]:
# ===== STEP 3: HVG SELECTION WITH FORCED MARKERS =====
print(f"\n{'='*80}")
print("STEP 3: HVG SELECTION WITH FORCED MARKERS")
print(f"{'='*80}\n")

# Define forced markers for secretory lineage
FORCED_MARKERS_SECRETORY = [
    # Secretory general
    "SCGB1A1", "SCGB3A1", "MUC5B", "BPIFA1", "BPIFB1",
    # Goblet cells
    "MUC5AC", "TFF3", "SPDEF", "AGR2", "FCGBP",
    # Serous cells
    "LTF", "LYZ", "DMBT1", "PIP", "AZGP1",
    # Club cells
    "SCGB1A1", "SCGB3A2", "CYP2F1", "CCKAR", "LYPD2",
    # Basal markers (for comparison)
    "KRT5", "TP63", "KRT14", "KRT15",
    # Ciliated (for context)
    "FOXJ1", "CAPS", "RSPH1", "TPPP3",
    # Stress response
    "HSP90AA1", "HSPA1A", "HSPA1B", "DNAJB1",
    # Proliferation
    "MKI67", "TOP2A", "PCNA", "CENPW",
]

print(f"Forced markers defined: {len(FORCED_MARKERS_SECRETORY)} genes")

# Function to force include markers in HVG
def force_include_markers_in_hvg(
    adata,
    n_top_genes: int,
    forced_markers: list,
    symbol_col: str = None,
    case_insensitive: bool = True,
    hvg_col: str = "highly_variable"
):
    """
    Force include biological markers into HVG set while maintaining target size.
    Strategy: add missing markers, then drop worst-ranked non-forced HVGs.
    
    Returns:
        Dictionary with statistics
    """
    if hvg_col not in adata.var.columns:
        raise ValueError(f"'{hvg_col}' not found. Run sc.pp.highly_variable_genes first.")
    
    # Match genes
    if symbol_col and symbol_col in adata.var.columns:
        gene_vec = adata.var[symbol_col].astype(str)
    else:
        gene_vec = adata.var_names.astype(str)
    
    if case_insensitive:
        gene_vec_cmp = gene_vec.str.upper()
        forced_cmp = set([str(x).upper() for x in forced_markers])
    else:
        gene_vec_cmp = gene_vec
        forced_cmp = set([str(x) for x in forced_markers])
    
    forced_mask = gene_vec_cmp.isin(forced_cmp).values
    present_forced = int(forced_mask.sum())
    
    hvg_mask = adata.var[hvg_col].values.astype(bool)
    original_hvg_n = int(hvg_mask.sum())
    
    # Add forced markers
    missing_forced_mask = forced_mask & (~hvg_mask)
    add_n = int(missing_forced_mask.sum())
    hvg_mask = hvg_mask | forced_mask
    
    # Trim to n_top_genes if exceeded
    if int(hvg_mask.sum()) > n_top_genes:
        # Use ranking to drop worst non-forced HVGs
        if "highly_variable_rank" in adata.var.columns:
            rank = adata.var["highly_variable_rank"].fillna(1e18).values.astype(float)
            sort_key = rank
        elif "variances_norm" in adata.var.columns:
            vn = adata.var["variances_norm"].fillna(-1e18).values.astype(float)
            sort_key = -vn
        else:
            print("⚠️  No HVG ranking column found; keeping expanded HVG set.")
            adata.var[hvg_col] = hvg_mask
            return {
                "original_hvg_n": original_hvg_n,
                "present_forced": present_forced,
                "added_forced": add_n,
                "final_hvg_n": int(hvg_mask.sum()),
                "trimmed": 0
            }
        
        candidates_drop = np.where(hvg_mask & (~forced_mask))[0]
        candidates_drop = candidates_drop[np.argsort(sort_key[candidates_drop])[::-1]]
        
        need_drop = int(hvg_mask.sum()) - n_top_genes
        drop_idx = candidates_drop[:need_drop]
        hvg_mask[drop_idx] = False
        
        trimmed = len(drop_idx)
    else:
        trimmed = 0
    
    adata.var[hvg_col] = hvg_mask
    
    return {
        "original_hvg_n": original_hvg_n,
        "present_forced": present_forced,
        "added_forced": add_n,
        "final_hvg_n": int(hvg_mask.sum()),
        "trimmed": trimmed
    }

# HVG parameters
N_TOP_GENES = 4000

# CRITICAL: Check data prerequisites
print(f"Pre-HVG checks:")
print(f"  Total genes in dataset: {adata.n_vars:,}")

if adata.n_vars < N_TOP_GENES:
    print(f"  ⚠️  WARNING: Total genes ({adata.n_vars:,}) < target HVG ({N_TOP_GENES:,})")
    print(f"     Adjusting target to {adata.n_vars:,}")
    N_TOP_GENES = adata.n_vars

# Determine which layer to use for HVG
hvg_layer = None
if 'counts' in adata.layers:
    if sparse.issparse(adata.layers['counts']):
        counts_max = adata.layers['counts'].data.max()
    else:
        counts_max = adata.layers['counts'].max()
    
    if counts_max > 50:
        hvg_layer = 'counts'
        print(f"  ✓ Using layers['counts'] for HVG (raw counts detected)")
    else:
        print(f"  ⚠️  WARNING: layers['counts'] max={counts_max:.1f}, doesn't look like raw counts")
        hvg_layer = 'counts'  # Try anyway
else:
    print(f"  ⚠️  No 'counts' layer, will use adata.X for HVG")
    hvg_layer = None

# Check if HVG already computed
if 'highly_variable' not in adata.var.columns:
    print(f"\nComputing highly variable genes (n_top={N_TOP_GENES:,})...")
    
    # Determine batch key for batch-aware HVG
    batch_key_candidates = ['dataset', 'batch', 'Sample', 'sample', 'sample_id', 'orig.ident']
    batch_key_hvg = None
    for candidate in batch_key_candidates:
        if candidate in adata.obs.columns:
            n_batches = adata.obs[candidate].nunique()
            if n_batches >= 2:
                batch_key_hvg = candidate
                print(f"  Detected batch key: '{batch_key_hvg}' ({n_batches} batches)")
                break
    
    # Run HVG with fallback
    try:
        if batch_key_hvg:
            print(f"  Attempting batch-aware HVG...")
            sc.pp.highly_variable_genes(
                adata,
                layer=hvg_layer,
                n_top_genes=N_TOP_GENES,
                batch_key=batch_key_hvg,
                subset=False
            )
            hvg_method = f"batch-aware ({batch_key_hvg})"
        else:
            print(f"  No batch key found, using standard HVG...")
            sc.pp.highly_variable_genes(
                adata,
                layer=hvg_layer,
                n_top_genes=N_TOP_GENES,
                subset=False
            )
            hvg_method = "standard"
    except Exception as e:
        print(f"  ⚠️  Batch-aware HVG failed: {e}")
        print(f"  Falling back to standard HVG...")
        sc.pp.highly_variable_genes(
            adata,
            layer=hvg_layer,
            n_top_genes=N_TOP_GENES,
            subset=False
        )
        hvg_method = "standard-fallback"
    
    print(f"✓ HVG computed ({hvg_method})")
else:
    print(f"✓ HVG already computed")
    hvg_method = "pre-existing"

# Check HVG results
n_hvg = adata.var['highly_variable'].sum()
print(f"\n  Initial HVG count: {n_hvg:,}")

if n_hvg < N_TOP_GENES * 0.75:
    print(f"  ⚠️  WARNING: HVG count ({n_hvg:,}) much lower than target ({N_TOP_GENES:,})")
    print(f"     Possible reasons:")
    print(f"     1. Total genes < target")
    print(f"     2. Many genes have zero/low variance")
    print(f"     3. Batch correction removed too many genes")
    print(f"     → Check if data quality is OK")
elif n_hvg < N_TOP_GENES:
    print(f"  Note: HVG count ({n_hvg:,}) slightly lower than target ({N_TOP_GENES:,})")
else:
    print(f"  ✓ HVG count matches target")

# Check what columns were created
hvg_columns = [col for col in adata.var.columns if 'highly' in col.lower() or 'variance' in col.lower()]
print(f"\n  HVG-related columns created: {hvg_columns}")

# Force include markers
print(f"\n{'='*80}")
print(f"Forcing inclusion of {len(FORCED_MARKERS_SECRETORY)} secretory markers...")

# Try to detect symbol column
symbol_col = None
for col in ['symbol', 'symbol_base', 'gene_symbol', 'feature_name']:
    if col in adata.var.columns:
        symbol_col = col
        print(f"  Using symbol column: '{col}'")
        break

if symbol_col is None:
    print(f"  No symbol column found, using var_names directly")

force_stats = force_include_markers_in_hvg(
    adata,
    n_top_genes=N_TOP_GENES,
    forced_markers=FORCED_MARKERS_SECRETORY,
    symbol_col=symbol_col,
    case_insensitive=True
)

print(f"\n✓ Forced marker inclusion completed:")
print(f"  Original HVG: {force_stats['original_hvg_n']:,}")
print(f"  Markers present: {force_stats['present_forced']}/{len(FORCED_MARKERS_SECRETORY)}")
print(f"  Markers added: {force_stats['added_forced']}")
print(f"  Non-markers trimmed: {force_stats['trimmed']}")
print(f"  Final HVG: {force_stats['final_hvg_n']:,}")

# Diagnostic: which markers were missing?
if force_stats['added_forced'] > 0:
    print(f"\n  Markers that were added (previously not in HVG):")
    # Find which markers were added
    if symbol_col:
        gene_vec = adata.var[symbol_col].str.upper()
    else:
        gene_vec = adata.var_names.str.upper()
    
    forced_upper = [m.upper() for m in FORCED_MARKERS_SECRETORY]
    forced_mask = gene_vec.isin(forced_upper)
    hvg_mask = adata.var['highly_variable']
    
    # Find markers in dataset
    markers_in_data = []
    for m in FORCED_MARKERS_SECRETORY:
        if gene_vec.str.contains(m.upper(), case=False).any():
            markers_in_data.append(m)
    
    print(f"    Total markers in dataset: {len(markers_in_data)}/{len(FORCED_MARKERS_SECRETORY)}")
    
    if force_stats['added_forced'] <= 10:
        # Show which ones
        added_markers = [m for m in markers_in_data if m.upper() in gene_vec[forced_mask & ~adata.var['highly_variable'].shift(1, fill_value=False)].str.upper().tolist()]
        if added_markers:
            print(f"    Added markers: {', '.join(added_markers[:10])}")

# Save HVG method to uns
adata.uns['hvg_method'] = hvg_method
adata.uns['hvg_n_top_genes'] = N_TOP_GENES
adata.uns['hvg_forced_markers'] = FORCED_MARKERS_SECRETORY
adata.uns['hvg_layer'] = hvg_layer if hvg_layer else 'X'

print(f"\n✓ HVG calculation complete")

## 4. Batch Key Detection and QC



## 5. Small Batch Filtering (CRITICAL)



## 6. Backup Original Embeddings



## 7. PCA Computation



In [ ]:
print(f"\n{'='*80}")
print("STEP 6: BACKUP ORIGINAL EMBEDDINGS")
print(f"{'='*80}\n")

# Backup existing UMAP if present
if 'X_umap' in adata.obsm:
    if 'X_umap_original' not in adata.obsm:
        adata.obsm['X_umap_original'] = np.asarray(adata.obsm['X_umap']).copy()
        print(f"✓ Backed up X_umap → X_umap_original")
    else:
        print(f"  X_umap_original already exists, skipping backup")

# Backup existing leiden if present
if 'leiden' in adata.obs.columns:
    if 'leiden_original' not in adata.obs.columns:
        adata.obs['leiden_original'] = adata.obs['leiden'].copy()
        print(f"✓ Backed up leiden → leiden_original")



## 7. PCA Computation (if needed)



In [ ]:
print(f"\n{'='*80}")
print("STEP 7: PCA COMPUTATION")
print(f"{'='*80}\n")

need_pca = (
    'X_pca' not in adata.obsm or 
    adata.obsm['X_pca'].shape[1] < BBKNN_CONFIG['n_pcs']
)

if need_pca:
    print(f"  Computing PCA with {BBKNN_CONFIG['n_pcs']} components...")
    
    # Use HVG if available
    use_hvg = (
        'highly_variable' in adata.var.columns and 
        np.any(adata.var['highly_variable'].values)
    )
    
    if use_hvg:
        n_hvg = adata.var['highly_variable'].sum()
        print(f"    Using {n_hvg:,} highly variable genes")
    
    try:
        sc.pp.pca(
            adata, 
            n_comps=BBKNN_CONFIG['n_pcs'], 
            svd_solver='arpack',
            use_highly_variable=use_hvg,
            random_state=42
        )
        print(f"✓ PCA computed")
    except Exception as e:
        raise RuntimeError(f"PCA failed: {e}")
else:
    print(f"✓ PCA already available ({adata.obsm['X_pca'].shape[1]} components)")



## 8. BBKNN Graph Construction



In [ ]:
print(f"\n{'='*80}")
print("STEP 8: BBKNN GRAPH CONSTRUCTION")
print(f"{'='*80}\n")

print(f"  Parameters:")
print(f"    batch_key: {BBKNN_CONFIG['batch_key']}")
print(f"    neighbors_within_batch: {BBKNN_CONFIG['neighbors_within_batch']}")
print(f"    n_pcs: {BBKNN_CONFIG['n_pcs']}")
print(f"    metric: {BBKNN_CONFIG['metric']}")

try:
    bbknn.bbknn(
        adata,
        batch_key=BBKNN_CONFIG['batch_key'],
        neighbors_within_batch=BBKNN_CONFIG['neighbors_within_batch'],
        n_pcs=BBKNN_CONFIG['n_pcs'],
        metric=BBKNN_CONFIG['metric'],
        trim=BBKNN_CONFIG['trim'],
        key_added='neighbors_bbknn',
        copy=False,
    )
    print(f"\n✓ BBKNN graph constructed successfully")
except Exception as e:
    raise RuntimeError(f"BBKNN failed: {e}")



## 9. UMAP Computation



In [ ]:
print(f"\n{'='*80}")
print("STEP 9: UMAP COMPUTATION")
print(f"{'='*80}\n")

try:
    sc.tl.umap(
        adata,
        neighbors_key='neighbors_bbknn',
        min_dist=BBKNN_CONFIG['umap_min_dist'],
        spread=BBKNN_CONFIG['umap_spread'],
        random_state=42,
    )
    print(f"✓ UMAP computed")
except Exception as e:
    raise RuntimeError(f"UMAP failed: {e}")

# Rename to distinguish from original
adata.obsm['X_umap_bbknn'] = adata.obsm['X_umap'].copy()



## 10. Leiden Clustering at Multiple Resolutions



In [ ]:
print(f"\n{'='*80}")
print("STEP 10: LEIDEN CLUSTERING")
print(f"{'='*80}\n")

for res in BBKNN_CONFIG['leiden_resolutions']:
    key = f'leiden_bbknn_res{res}'
    
    print(f"  Computing leiden at resolution {res}...")
    
    try:
        sc.tl.leiden(
            adata,
            neighbors_key='neighbors_bbknn',
            resolution=res,
            key_added=key,
            flavor='igraph',
            n_iterations=2,
            directed=False,
            random_state=42
        )
    except TypeError:
        # Fallback if flavor not supported
        sc.tl.leiden(
            adata,
            neighbors_key='neighbors_bbknn',
            resolution=res,
            key_added=key,
            n_iterations=2,
            random_state=42
        )
    
    n_clusters = adata.obs[key].nunique()
    print(f"    → {n_clusters} clusters")

# Set default leiden
default_key = f'leiden_bbknn_res{BBKNN_CONFIG["default_resolution"]}'
adata.obs['leiden_bbknn'] = adata.obs[default_key].copy()

print(f"\n✓ Clustering completed")
print(f"  Default clustering: leiden_bbknn ({adata.obs['leiden_bbknn'].nunique()} clusters)")



## 10. Leiden Clustering



## 9. UMAP Embedding



## 8. BBKNN Batch Correction



## 11. UMAP Visualization - Batch Integration Check



In [ ]:
print(f"\n{'='*80}")
print("STEP 11: BATCH INTEGRATION VISUALIZATION")
print(f"{'='*80}\n")

# Batch integration
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before BBKNN (if available)
if 'X_umap_original' in adata.obsm:
    sc.pl.embedding(
        adata, basis='umap_original', color=batch_key,
        ax=axes[0], show=False, title='Before BBKNN'
    )
else:
    axes[0].text(0.5, 0.5, 'Original UMAP not available',
                ha='center', va='center', fontsize=14)
    axes[0].axis('off')

# After BBKNN
sc.pl.embedding(
    adata, basis='umap_bbknn', color=batch_key,
    ax=axes[1], show=False, title='After BBKNN'
)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}batch_integration_comparison.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✓ Batch integration plot saved")



## 12. UMAP Visualization - Clustering Results



In [ ]:
print(f"\n{'='*80}")
print("STEP 12: CLUSTERING VISUALIZATION")
print(f"{'='*80}\n")

# Plot all leiden resolutions
leiden_keys = [f'leiden_bbknn_res{res}' for res in BBKNN_CONFIG['leiden_resolutions']]
n_plots = len(leiden_keys)

fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
if n_plots == 1:
    axes = [axes]

for i, key in enumerate(leiden_keys):
    res = BBKNN_CONFIG['leiden_resolutions'][i]
    n_clusters = adata.obs[key].nunique()
    
    sc.pl.embedding(
        adata, basis='umap_bbknn', color=key,
        ax=axes[i], show=False, legend_loc='on data',
        title=f'Leiden res={res} ({n_clusters} clusters)'
    )

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}leiden_clustering_resolutions.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✓ Clustering plots saved")



## 13. CellTypist Annotation Visualization (if available)



In [ ]:
print(f"\n{'='*80}")
print("STEP 13: CELLTYPIST ANNOTATION VISUALIZATION")
print(f"{'='*80}\n")

# Find CellTypist column
celltypist_col = None
for col in ['celltypist_pred','celltypist_majority_voting', 'cell_type', 'celltype', 'annotation']:
    if col in adata.obs.columns:
        celltypist_col = col
        print(f"✓ Found CellTypist annotation: '{col}'")
        break

if celltypist_col:
    # UMAP colored by CellTypist
    sc.pl.umap(
        adata, color=celltypist_col,
        frameon=False, title='CellTypist Annotation (BBKNN UMAP)',
        save='_bbknn_celltypist.pdf'
    )
    
    # Cell type composition
    celltypist_counts = adata.obs[celltypist_col].value_counts()
    print(f"\nCellTypist annotation distribution:")
    for ct, count in celltypist_counts.items():
        pct = count / adata.n_obs * 100
        print(f"  {ct}: {count:,} ({pct:.1f}%)")
else:
    print(f"⚠️  No CellTypist annotation found")



## 14. Marker Gene Expression Visualization



In [ ]:
# ===== STEP 14: MARKER GENE EXPRESSION =====
print(f"\n{'='*80}")
print("STEP 14: MARKER GENE EXPRESSION")
print(f"{'='*80}\n")

# Define secretory markers
secretory_markers = {
    'Secretory_General': ['SCGB1A1', 'SCGB3A1', 'MUC5B'],
    'Goblet': ['MUC5AC', 'TFF3', 'SPDEF'],
    'Serous': ['LTF', 'LYZ', 'DMBT1'],
    'Club': ['SCGB1A1', 'SCGB3A2', 'CYP2F1'],
    'Basal': ['KRT5', 'TP63', 'KRT14'],
    'Stress': ['HSP90AA1', 'HSPA1A', 'HSPA1B'],
}

# Flatten and check availability
all_markers = []
for markers in secretory_markers.values():
    all_markers.extend(markers)
all_markers = list(set(all_markers))

# CRITICAL: Determine correct layer for visualization
# Check what type of data is in .raw and .X
print(f"Checking data layers:")

if adata.raw is not None:
    # Check if .raw contains log-normalized or counts
    if sparse.issparse(adata.raw.X):
        raw_max = adata.raw.X.data.max()
    else:
        raw_max = adata.raw.X.max()
    
    print(f"  .raw.X max value: {raw_max:.1f}")
    
    if raw_max > 50:
        print(f"    → Appears to be counts (not log-normalized)")
        raw_is_log = False
    else:
        print(f"    → Appears to be log-normalized")
        raw_is_log = True
else:
    print(f"  .raw: Not available")
    raw_is_log = False

# Check current .X
if sparse.issparse(adata.X):
    x_max = adata.X.data.max()
else:
    x_max = adata.X.max()

print(f"  .X max value: {x_max:.1f}")
if x_max > 50:
    print(f"    → Appears to be counts (not log-normalized)")
    x_is_log = False
else:
    print(f"    → Appears to be log-normalized")
    x_is_log = True

# Decide visualization strategy
if adata.raw is not None and raw_is_log:
    # Use .raw if it's log-normalized
    available_markers = [g for g in all_markers if g in adata.raw.var_names]
    use_raw = True
    layer_to_use = None
    print(f"\n✓ Using .raw for visualization (log-normalized)")
elif x_is_log:
    # Use .X if it's log-normalized
    available_markers = [g for g in all_markers if g in adata.var_names]
    use_raw = False
    layer_to_use = None
    print(f"\n✓ Using .X for visualization (log-normalized)")
elif 'log1p' in adata.layers:
    # Fallback to log1p layer
    available_markers = [g for g in all_markers if g in adata.var_names]
    use_raw = False
    layer_to_use = 'log1p'
    print(f"\n✓ Using layers['log1p'] for visualization")
else:
    # Last resort: use .X and warn
    available_markers = [g for g in all_markers if g in adata.var_names]
    use_raw = False
    layer_to_use = None
    print(f"\n⚠️  WARNING: Data may not be log-normalized!")
    print(f"   Expression values may look wrong (very high range)")

print(f"\nMarker genes: {len(available_markers)}/{len(all_markers)} available")
if len(available_markers) < len(all_markers):
    missing = set(all_markers) - set(available_markers)
    print(f"  Missing: {missing}")

# UMAP with marker expression
if available_markers:
    # Plot in batches of 6
    for i in range(0, len(available_markers), 6):
        batch_markers = available_markers[i:i+6]
        sc.pl.umap(
            adata, color=batch_markers, ncols=3,
            use_raw=use_raw, layer=layer_to_use,
            frameon=False, cmap='RdBu_r',
            save=f'_bbknn_markers_batch{i//6+1}.pdf'
        )
    print(f"✓ Marker expression plots saved")

## 15. Dotplot: Markers × Clusters



In [ ]:
# ===== STEP 15: MARKER DOTPLOT =====
print(f"\n{'='*80}")
print("STEP 15: MARKER DOTPLOT")
print(f"{'='*80}\n")

if available_markers:
    # Check if dendrogram exists and is compatible
    need_dendrogram = False
    if 'dendrogram_leiden_bbknn' in adata.uns:
        # Check if it's compatible
        try:
            stored_categories = adata.uns['dendrogram_leiden_bbknn']['categories_idx_ordered']
            current_categories = adata.obs['leiden_bbknn'].cat.categories
            if len(stored_categories) == len(current_categories):
                print(f"✓ Compatible dendrogram found")
                need_dendrogram = False
            else:
                print(f"⚠️  Incompatible dendrogram (old: {len(stored_categories)}, new: {len(current_categories)})")
                need_dendrogram = True
        except:
            need_dendrogram = True
    else:
        need_dendrogram = True
    
    # Recompute dendrogram if needed
    if need_dendrogram:
        print(f"  Computing new dendrogram for leiden_bbknn...")
        try:
            sc.tl.dendrogram(adata, groupby='leiden_bbknn', use_rep='X_pca')
            print(f"✓ Dendrogram computed")
            use_dendrogram = True
        except Exception as e:
            print(f"⚠️  Dendrogram computation failed: {e}")
            print(f"   Will create dotplot without dendrogram")
            use_dendrogram = False
    else:
        use_dendrogram = True
    
    # Create dotplot
    try:
        sc.pl.dotplot(
            adata, available_markers, 
            groupby='leiden_bbknn',
            use_raw=use_raw,
            layer=layer_to_use,
            dendrogram=use_dendrogram,
            save='_bbknn_markers_by_cluster.pdf'
        )
        print(f"✓ Dotplot saved")
    except Exception as e:
        print(f"⚠️  Dotplot with dendrogram failed: {e}")
        print(f"   Retrying without dendrogram...")
        try:
            sc.pl.dotplot(
                adata, available_markers, 
                groupby='leiden_bbknn',
                use_raw=use_raw,
                layer=layer_to_use,
                dendrogram=False,
                save='_bbknn_markers_by_cluster_no_dendro.pdf'
            )
            print(f"✓ Dotplot saved (without dendrogram)")
        except Exception as e2:
            print(f"⚠️  Dotplot completely failed: {e2}")

## 16. Differential Expression: Find Markers per Cluster



In [ ]:
print(f"\n{'='*80}")
print("STEP 16: DIFFERENTIAL EXPRESSION - CLUSTER MARKERS")
print(f"{'='*80}\n")

print(f"Parameters:")
print(f"  Method: {MARKER_CONFIG['method']}")
print(f"  Min pct: {MARKER_CONFIG['min_pct']}")
print(f"  LogFC threshold: {MARKER_CONFIG['logfc_threshold']}")
print(f"  Use raw: {MARKER_CONFIG['use_raw'] and adata.raw is not None}")

# Check if DE already done
if 'rank_genes_groups' in adata.uns:
    print(f"\n⚠️  Previous DE results found, will overwrite")

# Run DE
try:
    use_raw_de = MARKER_CONFIG['use_raw'] and adata.raw is not None
    
    sc.tl.rank_genes_groups(
        adata,
        groupby='leiden_bbknn',
        method=MARKER_CONFIG['method'],
        use_raw=use_raw_de,
        pts=True,  # Calculate percentage expressed
        key_added='rank_genes_groups',
        n_genes=200,  # Get more genes for downstream analysis
    )
    print(f"\n✓ Differential expression completed")
    
except Exception as e:
    raise RuntimeError(f"Differential expression failed: {e}")

# Extract top markers per cluster
print(f"\nTop {MARKER_CONFIG['top_n']} markers per cluster:")

results_dict = {
    'cluster': [],
    'gene': [],
    'logfoldchanges': [],
    'pvals_adj': [],
    'pct_in_cluster': [],
    'pct_out_cluster': [],
}

for cluster in adata.obs['leiden_bbknn'].cat.categories:
    de_result = sc.get.rank_genes_groups_df(
        adata, 
        group=cluster,
        key='rank_genes_groups'
    )
    
    # Filter by thresholds
    de_filtered = de_result[
        (de_result['pvals_adj'] < 0.05) &
        (de_result['logfoldchanges'] > MARKER_CONFIG['logfc_threshold']) &
        (de_result['pct_nz_group'] > MARKER_CONFIG['min_pct'])
    ].head(MARKER_CONFIG['top_n'])
    
    print(f"\n  Cluster {cluster}: {len(de_filtered)} significant markers")
    if len(de_filtered) > 0:
        print(f"    Top 3: {', '.join(de_filtered['names'].head(3).tolist())}")
    
    # Store results
    for _, row in de_filtered.iterrows():
        results_dict['cluster'].append(cluster)
        results_dict['gene'].append(row['names'])
        results_dict['logfoldchanges'].append(row['logfoldchanges'])
        results_dict['pvals_adj'].append(row['pvals_adj'])
        results_dict['pct_in_cluster'].append(row['pct_nz_group'])
        results_dict['pct_out_cluster'].append(row['pct_nz_reference'])

# Save to DataFrame
markers_df = pd.DataFrame(results_dict)
markers_df.to_csv(f'{OUTPUT_DIR}cluster_markers_top{MARKER_CONFIG["top_n"]}.csv', index=False)
print(f"\n✓ Markers saved to cluster_markers_top{MARKER_CONFIG['top_n']}.csv")



In [ ]:
# ===== Save All Cluster Markers (完整版) =====
print(f"\n{'='*80}")
print(f"Saving complete marker gene results...")
print(f"{'='*80}\n")

try:
    # Extract ALL markers for each cluster (not just top N)
    all_cluster_markers = []
    
    for cluster in adata.obs['leiden_bbknn'].cat.categories:
        print(f"  Extracting markers for cluster {cluster}...")
        
        de_result = sc.get.rank_genes_groups_df(
            adata, 
            group=cluster,
            key='rank_genes_groups'
        )
        
        # Add cluster column
        de_result['cluster'] = cluster
        
        # Reorder columns for readability
        cols = ['cluster', 'names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj', 
                'pct_nz_group', 'pct_nz_reference']
        # Only keep columns that exist
        cols = [c for c in cols if c in de_result.columns]
        de_result = de_result[cols]
        
        all_cluster_markers.append(de_result)
    
    # Combine all results
    all_markers_df = pd.concat(all_cluster_markers, ignore_index=True)
    
    # Sort by cluster and adjusted p-value
    all_markers_df = all_markers_df.sort_values(['cluster', 'pvals_adj'])
    
    # Save complete results
    output_file = f'{OUTPUT_DIR}all_cluster_markers_complete.csv'
    all_markers_df.to_csv(output_file, index=False)
    
    print(f"\n✓ Complete marker results saved:")
    print(f"  File: {output_file}")
    print(f"  Total genes tested: {len(all_markers_df):,}")
    print(f"  Clusters: {adata.obs['leiden_bbknn'].nunique()}")
    print(f"  Columns: {list(all_markers_df.columns)}")
    
    # Summary statistics
    print(f"\n  Summary per cluster:")
    for cluster in adata.obs['leiden_bbknn'].cat.categories:
        cluster_data = all_markers_df[all_markers_df['cluster'] == cluster]
        sig_genes = cluster_data[cluster_data['pvals_adj'] < 0.05]
        high_fc = sig_genes[sig_genes['logfoldchanges'] > MARKER_CONFIG['logfc_threshold']]
        
        print(f"    Cluster {cluster}: {len(cluster_data):,} genes total, "
              f"{len(sig_genes):,} significant (adj.p<0.05), "
              f"{len(high_fc):,} high FC (>{MARKER_CONFIG['logfc_threshold']})")
    
    # Also save a filtered version (significant only)
    sig_markers = all_markers_df[
        (all_markers_df['pvals_adj'] < 0.05) &
        (all_markers_df['logfoldchanges'] > MARKER_CONFIG['logfc_threshold'])
    ]
    
    sig_output_file = f'{OUTPUT_DIR}all_cluster_markers_significant.csv'
    sig_markers.to_csv(sig_output_file, index=False)
    
    print(f"\n✓ Significant markers saved:")
    print(f"  File: {sig_output_file}")
    print(f"  Significant genes: {len(sig_markers):,} "
          f"({len(sig_markers)/len(all_markers_df)*100:.1f}% of total)")
    
except Exception as e:
    print(f"❌ Failed to save all markers: {e}")
    import traceback
    traceback.print_exc()

## 17. Differential Expression: CellTypist Annotation Markers



In [ ]:
print(f"\n{'='*80}")
print("STEP 17: DIFFERENTIAL EXPRESSION - CELLTYPIST MARKERS")
print(f"{'='*80}\n")

if celltypist_col:
    print(f"Finding markers for CellTypist annotation: {celltypist_col}")
    
    try:
        use_raw_de = MARKER_CONFIG['use_raw'] and adata.raw is not None
        
        sc.tl.rank_genes_groups(
            adata,
            groupby=celltypist_col,
            method=MARKER_CONFIG['method'],
            use_raw=use_raw_de,
            pts=True,
            key_added='rank_genes_celltypist',
            n_genes=200,
        )
        print(f"✓ DE by CellTypist annotation completed")
        
        # Extract top markers
        print(f"\nTop {MARKER_CONFIG['top_n']} markers per cell type:")
        
        celltypist_results = {
            'cell_type': [],
            'gene': [],
            'logfoldchanges': [],
            'pvals_adj': [],
            'pct_in_type': [],
            'pct_out_type': [],
        }
        
        for cell_type in adata.obs[celltypist_col].unique():
            if pd.isna(cell_type):
                continue
                
            de_result = sc.get.rank_genes_groups_df(
                adata,
                group=cell_type,
                key='rank_genes_celltypist'
            )
            
            # Filter
            de_filtered = de_result[
                (de_result['pvals_adj'] < 0.05) &
                (de_result['logfoldchanges'] > MARKER_CONFIG['logfc_threshold']) &
                (de_result['pct_nz_group'] > MARKER_CONFIG['min_pct'])
            ].head(MARKER_CONFIG['top_n'])
            
            print(f"\n  {cell_type}: {len(de_filtered)} significant markers")
            if len(de_filtered) > 0:
                print(f"    Top 3: {', '.join(de_filtered['names'].head(3).tolist())}")
            
            # Store
            for _, row in de_filtered.iterrows():
                celltypist_results['cell_type'].append(cell_type)
                celltypist_results['gene'].append(row['names'])
                celltypist_results['logfoldchanges'].append(row['logfoldchanges'])
                celltypist_results['pvals_adj'].append(row['pvals_adj'])
                celltypist_results['pct_in_type'].append(row['pct_nz_group'])
                celltypist_results['pct_out_type'].append(row['pct_nz_reference'])
        
        # Save
        celltypist_markers_df = pd.DataFrame(celltypist_results)
        celltypist_markers_df.to_csv(
            f'{OUTPUT_DIR}celltypist_markers_top{MARKER_CONFIG["top_n"]}.csv',
            index=False
        )
        print(f"\n✓ CellTypist markers saved")
        
    except Exception as e:
        print(f"⚠️  CellTypist DE failed: {e}")
else:
    print(f"⚠️  Skipping (no CellTypist annotation)")



## 18. Heatmap Visualization of Top Markers



In [ ]:
print(f"\n{'='*80}")
print("STEP 18: MARKER HEATMAP VISUALIZATION")
print(f"{'='*80}\n")

# Cluster markers heatmap
try:
    sc.pl.rank_genes_groups_heatmap(
        adata,
        n_genes=10,
        groupby='leiden_bbknn',
        key='rank_genes_groups',
        use_raw=use_raw_de,
        show_gene_labels=True,
        dendrogram=True,
        swap_axes=False,
        save='_cluster_markers_heatmap.pdf'
    )
    print(f"✓ Cluster markers heatmap saved")
except Exception as e:
    print(f"⚠️  Cluster heatmap failed: {e}")

# CellTypist markers heatmap
if celltypist_col and 'rank_genes_celltypist' in adata.uns:
    try:
        sc.pl.rank_genes_groups_heatmap(
            adata,
            n_genes=10,
            groupby=celltypist_col,
            key='rank_genes_celltypist',
            use_raw=use_raw_de,
            show_gene_labels=True,
            dendrogram=True,
            swap_axes=False,
            save='_celltypist_markers_heatmap.pdf'
        )
        print(f"✓ CellTypist markers heatmap saved")
    except Exception as e:
        print(f"⚠️  CellTypist heatmap failed: {e}")



In [ ]:
# ===== STEP 18: MARKER HEATMAP VISUALIZATION =====
print(f"\n{'='*80}")
print("STEP 18: MARKER HEATMAP VISUALIZATION")
print(f"{'='*80}\n")

# Determine which data to use for heatmap
print(f"Configuring heatmap data source:")

# CRITICAL: Heatmap must use SAME data source as DE analysis
# Check what was used in Step 16
if 'use_raw_de' in locals():
    use_raw_heatmap = use_raw_de
    print(f"  Using same setting as DE analysis: use_raw={use_raw_heatmap}")
else:
    # Fallback: check if .raw exists and looks reasonable
    if adata.raw is not None:
        if sparse.issparse(adata.raw.X):
            raw_max = adata.raw.X.data.max()
        else:
            raw_max = adata.raw.X.max()
        
        if raw_max > 50:
            print(f"  ⚠️  .raw appears to be counts (max={raw_max:.0f})")
            print(f"     But using it anyway to match gene names")
            use_raw_heatmap = True  # Force use .raw for gene name consistency
        else:
            print(f"  ✓ .raw is log-normalized (max={raw_max:.1f})")
            use_raw_heatmap = True
    else:
        print(f"  ✓ Using .X for heatmap (no .raw available)")
        use_raw_heatmap = False

# CRITICAL: Use standard_scale to normalize across genes
print(f"  Scaling strategy: standard_scale='var' (Z-score per gene)")
print(f"  This ensures all genes are on comparable scale")
print(f"  Color range will be Z-score: -2 to +2\n")

# ===== Cluster markers heatmap =====
print(f"Creating cluster markers heatmap...")

try:
    sc.pl.rank_genes_groups_heatmap(
        adata,
        n_genes=10,
        groupby='leiden_bbknn',
        key='rank_genes_groups',
        use_raw=use_raw_heatmap,  # MUST match DE analysis
        show_gene_labels=True,
        dendrogram=True,
        standard_scale='var',  # CRITICAL: Z-score normalization
        swap_axes=False,
        cmap='RdBu_r',         # Better colormap for scaled data
        vmin=-2, vmax=2,       # Standard z-score range
        save='_cluster_markers_heatmap.pdf'
    )
    print(f"✓ Cluster markers heatmap saved")
    
except KeyError as e:
    error_msg = str(e)
    print(f"⚠️  KeyError in cluster heatmap: {e}")
    
    # Diagnose the issue
    if "Could not find keys" in error_msg:
        print(f"\n   DIAGNOSIS: Gene name mismatch!")
        print(f"   - DE analysis found genes in one naming system")
        print(f"   - But heatmap can't find them in current adata")
        print(f"\n   Trying alternative: force use_raw={not use_raw_heatmap}...")
        
        try:
            sc.pl.rank_genes_groups_heatmap(
                adata,
                n_genes=10,
                groupby='leiden_bbknn',
                key='rank_genes_groups',
                use_raw=not use_raw_heatmap,  # Try opposite
                show_gene_labels=True,
                dendrogram=True,
                standard_scale='var',
                swap_axes=False,
                cmap='RdBu_r',
                vmin=-2, vmax=2,
                save='_cluster_markers_heatmap.pdf'
            )
            print(f"✓ Cluster markers heatmap saved (with use_raw={not use_raw_heatmap})")
        except Exception as e2:
            print(f"   Still failed: {e2}")
            print(f"\n   Trying manual approach without missing genes...")
    else:
        print(f"   Possible cause: 'rank_genes_groups' key not found")
        print(f"   Make sure Step 16 (DE analysis) ran successfully")
    
except ValueError as e:
    print(f"⚠️  ValueError in cluster heatmap: {e}")
    print(f"   Retrying without dendrogram...")
    
    try:
        sc.pl.rank_genes_groups_heatmap(
            adata,
            n_genes=10,
            groupby='leiden_bbknn',
            key='rank_genes_groups',
            use_raw=use_raw_heatmap,
            show_gene_labels=True,
            dendrogram=False,  # Disable dendrogram
            standard_scale='var',
            swap_axes=False,
            cmap='RdBu_r',
            vmin=-2, vmax=2,
            save='_cluster_markers_heatmap_no_dendro.pdf'
        )
        print(f"✓ Cluster markers heatmap saved (without dendrogram)")
    except Exception as e2:
        print(f"❌ Cluster heatmap completely failed: {e2}")
        
except Exception as e:
    print(f"❌ Cluster heatmap failed with unexpected error: {e}")

# If all attempts failed, try manual heatmap
if not any(['_cluster_markers_heatmap.pdf' in str(e) for e in [None]]):  # Check if succeeded
    print(f"\n   Trying manual heatmap approach...")
    
    try:
        print(f"   Creating manual heatmap from available genes...")
        
        # Get top markers per cluster, filtering for available genes
        top_genes = []
        missing_genes = []
        
        # Determine which gene list to check against
        if use_raw_heatmap and adata.raw is not None:
            available_genes = set(adata.raw.var_names)
        else:
            available_genes = set(adata.var_names)
        
        for cluster in adata.obs['leiden_bbknn'].cat.categories:
            try:
                de_result = sc.get.rank_genes_groups_df(
                    adata, 
                    group=cluster,
                    key='rank_genes_groups'
                )
                
                # Filter for available genes
                for gene in de_result['names'].head(10):
                    if gene in available_genes:
                        top_genes.append(gene)
                    else:
                        missing_genes.append(gene)
                        
            except Exception as e:
                print(f"   Warning: Could not get markers for cluster {cluster}")
        
        top_genes = list(dict.fromkeys(top_genes))  # Remove duplicates, keep order
        
        if len(top_genes) > 0:
            print(f"   Found {len(top_genes)} available marker genes")
            if len(missing_genes) > 0:
                print(f"   Skipped {len(set(missing_genes))} missing genes")
            
            # Create manual heatmap
            sc.pl.heatmap(
                adata,
                var_names=top_genes[:100],  # Limit to 100 genes
                groupby='leiden_bbknn',
                use_raw=use_raw_heatmap,
                standard_scale='var',
                cmap='RdBu_r',
                vmin=-2, vmax=2,
                dendrogram=True,
                swap_axes=True,
                save='_cluster_markers_manual_heatmap.pdf'
            )
            print(f"✓ Manual cluster markers heatmap saved")
        else:
            print(f"❌ No available marker genes found for manual heatmap")
            
    except Exception as e3:
        print(f"❌ Manual heatmap also failed: {e3}")

# ===== CellTypist markers heatmap =====
if celltypist_col and 'rank_genes_celltypist' in adata.uns:
    print(f"\nCreating CellTypist markers heatmap...")
    
    try:
        sc.pl.rank_genes_groups_heatmap(
            adata,
            n_genes=10,
            groupby=celltypist_col,
            key='rank_genes_celltypist',
            use_raw=use_raw_heatmap,  # MUST match DE analysis
            show_gene_labels=True,
            dendrogram=True,
            standard_scale='var',  # CRITICAL: Z-score normalization
            swap_axes=False,
            cmap='RdBu_r',
            vmin=-2, vmax=2,
            save='_celltypist_markers_heatmap.pdf'
        )
        print(f"✓ CellTypist markers heatmap saved")
        
    except KeyError as e:
        print(f"⚠️  KeyError in CellTypist heatmap: {e}")
        print(f"   Trying with opposite use_raw setting...")
        
        try:
            sc.pl.rank_genes_groups_heatmap(
                adata,
                n_genes=10,
                groupby=celltypist_col,
                key='rank_genes_celltypist',
                use_raw=not use_raw_heatmap,  # Try opposite
                show_gene_labels=True,
                dendrogram=True,
                standard_scale='var',
                swap_axes=False,
                cmap='RdBu_r',
                vmin=-2, vmax=2,
                save='_celltypist_markers_heatmap.pdf'
            )
            print(f"✓ CellTypist markers heatmap saved (with use_raw={not use_raw_heatmap})")
        except Exception as e2:
            print(f"❌ CellTypist heatmap still failed: {e2}")
            
    except ValueError as e:
        print(f"⚠️  ValueError in CellTypist heatmap: {e}")
        print(f"   Retrying without dendrogram...")
        
        try:
            sc.pl.rank_genes_groups_heatmap(
                adata,
                n_genes=10,
                groupby=celltypist_col,
                key='rank_genes_celltypist',
                use_raw=use_raw_heatmap,
                show_gene_labels=True,
                dendrogram=False,
                standard_scale='var',
                swap_axes=False,
                cmap='RdBu_r',
                vmin=-2, vmax=2,
                save='_celltypist_markers_heatmap_no_dendro.pdf'
            )
            print(f"✓ CellTypist markers heatmap saved (without dendrogram)")
        except Exception as e2:
            print(f"❌ CellTypist heatmap completely failed: {e2}")
            
    except Exception as e:
        print(f"❌ CellTypist heatmap failed: {e}")
else:
    if not celltypist_col:
        print(f"\n⚠️  No CellTypist annotation found, skipping CellTypist heatmap")
    else:
        print(f"\n⚠️  'rank_genes_celltypist' not found, skipping CellTypist heatmap")
        print(f"   Make sure Step 17 (CellTypist DE) ran successfully")

print(f"\n{'='*80}")
print(f"✓ Heatmap visualization step completed")
print(f"{'='*80}")

## 19. Cluster-CellTypist Correspondence Analysis



In [ ]:
print(f"\n{'='*80}")
print("STEP 19: CLUSTER-CELLTYPIST CORRESPONDENCE")
print(f"{'='*80}\n")

if celltypist_col:
    # Crosstab
    correspondence = pd.crosstab(
        adata.obs['leiden_bbknn'],
        adata.obs[celltypist_col],
        margins=True
    )
    
    print(f"Cluster × CellTypist contingency table:")
    print(correspondence)
    
    # Save
    correspondence.to_csv(f'{OUTPUT_DIR}cluster_celltypist_correspondence.csv')
    print(f"\n✓ Correspondence table saved")
    
    # Heatmap
    correspondence_prop = correspondence.div(correspondence.sum(axis=1), axis=0).iloc[:-1, :-1]
    
    plt.figure(figsize=(max(10, len(correspondence_prop.columns)), 
                        max(6, len(correspondence_prop))))
    sns.heatmap(
        correspondence_prop,
        annot=True, fmt='.2f', cmap='YlOrRd',
        cbar_kws={'label': 'Proportion'},
        xticklabels=True, yticklabels=True
    )
    plt.title('Cluster → CellTypist Correspondence')
    plt.xlabel('CellTypist Annotation')
    plt.ylabel('BBKNN Leiden Cluster')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}cluster_celltypist_heatmap.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f"✓ Correspondence heatmap saved")
else:
    print(f"⚠️  Skipping (no CellTypist annotation)")



In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 假设你的 AnnData 对象是 `adata`

# 定义 marker 基因列表
markers = {
    'goblet': ['MUC5AC', 'SPDEF', 'AGR2', 'CLCA1', 'BPIFB1'],
    'ciliated': ['FOXJ1', 'PIFO', 'TPPP3', 'DNAH5'],
    'basal': ['KRT5', 'KRT14', 'TP63', 'KRT15'],
    'squamous': ['KRT4', 'KRT13', 'IVL', 'SPRR1', 'SPRR2'],
    'IFN': ['ISG15', 'IFIT3', 'MX1', 'OAS1'],
    'myeloid': ['LST1', 'TYROBP', 'FCER1G', 'S100A8', 'S100A9']
}

# 对于每个类别的 marker 基因，绘制 DotPlot
for marker_list in markers.values():
    sc.pl.dotplot(adata, marker_list, groupby='leiden_bbknn_res1.5', title=f'Marker Expression: {", ".join(marker_list)}')
    plt.show()


In [ ]:
# 计算每个 cluster 的 QC 统计量
qc_metrics = adata.obs.groupby('leiden_bbknn_res1.5').agg({
    'percent.mt': 'mean',
    'nFeature': 'mean',
    'nCount': 'mean'
})

# 打印出每个 cluster 的 QC 信息
print(qc_metrics)

# 可视化 QC 统计量
qc_metrics[['percent.mt', 'nFeatures', 'nCounts']].plot(kind='bar', figsize=(10, 6))
plt.title('Cluster-wise QC Metrics')
plt.ylabel('Mean Values')
plt.show()


In [ ]:
# 假设你想检查 cluster 0 中的 EPCAM 和 FOXJ1 表达
cluster_0 = adata[adata.obs['leiden_bbknn_res1.5'] == '0']

# 提取 EPCAM 和 FOXJ1 的表达数据
expressions = cluster_0[:, ['EPCAM', 'FOXJ1']].X

# 绘制 EPCAM vs FOXJ1 的散点图
import seaborn as sns
sns.scatterplot(x=expressions[:, 0], y=expressions[:, 1])
plt.title('EPCAM vs FOXJ1 expression in Cluster 0')
plt.xlabel('EPCAM expression')
plt.ylabel('FOXJ1 expression')
plt.show()


## 20. Cell Composition Analysis



In [ ]:
print(f"\n{'='*80}")
print("STEP 20: CELL COMPOSITION ANALYSIS")
print(f"{'='*80}\n")

# Cluster composition per batch
composition = pd.crosstab(
    adata.obs[batch_key],
    adata.obs['leiden_bbknn']
)

composition_prop = composition.div(composition.sum(axis=1), axis=0)

print(f"Cluster composition by batch:")
print(composition)

# Heatmap
plt.figure(figsize=(max(10, len(composition_prop.columns)), 
                    max(6, len(composition_prop))))
sns.heatmap(
    composition_prop,
    annot=True, fmt='.2f', cmap='YlGnBu',
    cbar_kws={'label': 'Proportion'}
)
plt.title(f'Cluster Composition by {batch_key}')
plt.xlabel('BBKNN Leiden Cluster')
plt.ylabel(batch_key)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}cluster_composition_by_batch.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✓ Composition analysis saved")



## 21. Save Processed Data



In [ ]:
print(f"\n{'='*80}")
print("STEP 21: SAVE PROCESSED DATA")
print(f"{'='*80}\n")

# Save h5ad
output_h5ad = f'{OUTPUT_DIR}Secretory_Lineage_bbknn_analyzed.h5ad'
adata.write_h5ad(output_h5ad, compression='gzip', compression_opts=9)
print(f"✓ Analyzed data saved: {output_h5ad}")
print(f"  Size: {os.path.getsize(output_h5ad) / 1e6:.1f} MB")



## 22. Generate Analysis Summary Report



In [ ]:
# Initialize marker dataframes if not already defined
if 'markers_df' not in locals():
    markers_df = pd.DataFrame()  # Empty dataframe as fallback
if 'celltypist_markers_df' not in locals():
    celltypist_markers_df = pd.DataFrame()  # Empty dataframe as fallback

print(f"\n{'='*80}")
print("STEP 22: ANALYSIS SUMMARY REPORT")
print(f"{'='*80}\n")

# Generate comprehensive report
report_lines = []
report_lines.append("="*80)
report_lines.append("SECRETORY LINEAGE BBKNN ANALYSIS - SUMMARY REPORT")
report_lines.append("="*80)
report_lines.append(f"\nAnalysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Author: r2end")
report_lines.append(f"\nInput: {INPUT_FILE}")
report_lines.append(f"Output: {OUTPUT_DIR}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("DATASET SUMMARY")
report_lines.append(f"{'-'*80}")
report_lines.append(f"Total cells: {adata.n_obs:,}")
report_lines.append(f"Total genes: {adata.n_vars:,}")
report_lines.append(f"Batch key: {batch_key}")
report_lines.append(f"Number of batches: {adata.obs[batch_key].nunique()}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("BBKNN PARAMETERS")
report_lines.append(f"{'-'*80}")
for key, value in BBKNN_CONFIG.items():
    report_lines.append(f"{key}: {value}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("CLUSTERING RESULTS")
report_lines.append(f"{'-'*80}")
for res in BBKNN_CONFIG['leiden_resolutions']:
    key = f'leiden_bbknn_res{res}'
    n_clusters = adata.obs[key].nunique()
    report_lines.append(f"Resolution {res}: {n_clusters} clusters")

report_lines.append(f"\nDefault clustering (leiden_bbknn):")
for cluster, count in adata.obs['leiden_bbknn'].value_counts().sort_index().items():
    pct = count / adata.n_obs * 100
    report_lines.append(f"  Cluster {cluster}: {count:,} cells ({pct:.1f}%)")

if celltypist_col:
    report_lines.append(f"\n{'-'*80}")
    report_lines.append("CELLTYPIST ANNOTATION")
    report_lines.append(f"{'-'*80}")
    for ct, count in adata.obs[celltypist_col].value_counts().items():
        pct = count / adata.n_obs * 100
        report_lines.append(f"{ct}: {count:,} cells ({pct:.1f}%)")

report_lines.append(f"\n{'-'*80}")
report_lines.append("DIFFERENTIAL EXPRESSION")
report_lines.append(f"{'-'*80}")
report_lines.append(f"Method: {MARKER_CONFIG['method']}")
report_lines.append(f"Min pct threshold: {MARKER_CONFIG['min_pct']}")
report_lines.append(f"LogFC threshold: {MARKER_CONFIG['logfc_threshold']}")
report_lines.append(f"Top N markers: {MARKER_CONFIG['top_n']}")

if "markers_df" in locals() and len(markers_df) > 0:
    report_lines.append(f"\nTotal significant markers (cluster): {len(markers_df)}")
    report_lines.append(f"Average markers per cluster: {len(markers_df) / adata.obs['leiden_bbknn'].nunique():.1f}")

if celltypist_col and "celltypist_markers_df" in locals() and len(celltypist_markers_df) > 0:
    report_lines.append(f"Total significant markers (CellTypist): {len(celltypist_markers_df)}")
    report_lines.append(f"Average markers per cell type: {len(celltypist_markers_df) / adata.obs[celltypist_col].nunique():.1f}")

report_lines.append(f"\n{'-'*80}")
report_lines.append("OUTPUT FILES")
report_lines.append(f"{'-'*80}")
report_lines.append(f"1. {output_h5ad}")
report_lines.append(f"2. {OUTPUT_DIR}cluster_markers_top{MARKER_CONFIG['top_n']}.csv")
if celltypist_col:
    report_lines.append(f"3. {OUTPUT_DIR}celltypist_markers_top{MARKER_CONFIG['top_n']}.csv")
    report_lines.append(f"4. {OUTPUT_DIR}cluster_celltypist_correspondence.csv")
report_lines.append(f"5. Multiple PDF figures in {OUTPUT_DIR}")

report_lines.append(f"\n{'='*80}")
report_lines.append("ANALYSIS COMPLETE")
report_lines.append(f"{'='*80}\n")

report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open(f'{OUTPUT_DIR}analysis_summary_report.txt', 'w') as f:
    f.write(report_text)

print(f"✓ Summary report saved: {OUTPUT_DIR}analysis_summary_report.txt")



## 23. Next Steps Recommendations



In [ ]:
print(f"\n{'='*80}")
print("RECOMMENDED NEXT STEPS")
print(f"{'='*80}\n")

print("""
Based on this BBKNN analysis, consider:

1. **Cluster Annotation Refinement**:
   - Review marker genes per cluster
   - Compare with CellTypist predictions
   - Manual curation of ambiguous clusters

2. **Trajectory Analysis**:
   - Identify differentiation trajectories
   - Pseudotime analysis with Monocle3
   - RNA velocity if spliced/unspliced data available

3. **Functional Analysis**:
   - Gene ontology enrichment per cluster
   - Pathway analysis for secretory subtypes
   - Cell-cell interaction prediction

4. **Cross-Tissue/Disease Comparison**:
   - Compare composition across conditions
   - Identify disease-associated cell states
   - Differential abundance testing

5. **Integration with Other Modalities**:
   - Spatial transcriptomics overlay
   - Protein expression validation
   - Regulatory network inference

For detailed analysis workflows, refer to:
- QUICK_REFERENCE_MEMORY.md
- Project documentation
""")

print(f"\n{'='*80}")
print("ALL ANALYSIS COMPLETE")
print(f"{'='*80}\n")

